# Lesson 11: k nearest neighbours with k = 101, a warning at a vote share of 0.2

The term spread is the 10-year Treasury yield minus the 3-month bill rate. A month is placed by its spread and the 12-month change of the bill rate, both standardised with the training months. Among the 101 nearest training months, the share followed by a US recession 12 months later is the vote share p. A month gets a warning when p is at least $t = 0.2$. A missed recession costs four times as much as a false alarm, so a warning has the lower expected cost from $p = 1/(1 + 4) = 0.2$ on. The neighbours are among the 561 months April 1953 to December 1999, and k = 101 has the highest mean AUC over three forward folds of these months. The warnings are counted on the 297 months January 2000 to September 2024.

Treasury yields and recession dates: Federal Reserve Board H.15 and NBER via FRED, Federal Reserve Bank of St. Louis. Snapshot 30 September 2026.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, roc_auc_score

months = pd.read_csv("https://digital-ai-finance.github.io/teaching-msc-data-science/data/2026-09-30-us-recession-monthly.csv", index_col="month", parse_dates=True)
train, test = months[months.split == "train"], months[months.split == "test"]
features, k = ["spread", "tbill_change_12m"], 101  # k = 101: the highest mean AUC over three forward folds
model = make_pipeline(StandardScaler(), KNeighborsClassifier(k)).fit(train[features], train.recession_in_12m)
p = model.predict_proba(test[features])[:, 1]
print(f"k = {k} nearest of the {len(train)} training months per test month")

plt.plot(test.index, p, color="black", label="vote share p")
plt.fill_between(test.index, 0, 1, where=test.recession_in_12m == 1, color="tab:red", alpha=0.2, label="recession 12 months later")
plt.axhline(0.2, ls="--", color="grey", label="threshold t = 0.2")
plt.xlabel("test month (year)")
plt.ylabel("vote share p of the 101 nearest months")
plt.legend(loc="upper left", bbox_to_anchor=(1, 1))
plt.show()

In [ ]:
t = 0.2  # a miss costs four false alarms: t = 1/(1 + 4)
tn, fp, fn, tp = confusion_matrix(test.recession_in_12m, p >= t).ravel()
print(f"warnings {tp + fp}: hits {tp}, misses {fn}, false alarms {fp}, calm months left calm {tn}")
print(f"cost at a miss of 24, a false alarm of 6: {24 * fn + 6 * fp}")
print(f"AUC on the test months: {roc_auc_score(test.recession_in_12m, p):.3f}")

Change `t = 0.2` to `t = 0.5` and run the cell again. At 0.5 no test month gets a warning, since every vote share is below 0.5, and the cost grows from 630 to 672, the cost of never warning.